# View plate.zarr in napari

Two ways to look at a plate.zarr store you've already generated (via `blimp convert
tiff`/`blimp convert nd2`'s NGFF output): a fast, full-resolution overview of the whole
plate laid out at its true row/column positions, and a detailed per-well view with
segmentation labels, point-object tables, and FOV boundaries.

Set `PLATE_PATH` in the cell below to point at your own plate.zarr -- this notebook
doesn't ship with one.

Needs `blimp` importable alongside napari and ngio -- from the `napari-feature-classifier`
environment, run from wherever your own blimp checkout lives:

```
pip install -e /path/to/blimp --no-deps
```

In [1]:
from pathlib import Path

import napari
from ngio import open_ome_zarr_plate, open_ome_zarr_container

from blimp.napari_utils import add_blimp_napari_methods

PLATE_PATH = Path("/Users/z3532965/Images/20260626_POLR2A_heterogeneity/20260626_POLR2A_heterogeneity.zarr")

plate = open_ome_zarr_plate(store=str(PLATE_PATH), mode="r")
wells = plate.wells_paths()
print("Wells with data:", wells)

WELL = wells[0]  # change to e.g. "C/09" to pick a different well
print("Viewing well:", WELL)

Wells with data: ['E/03', 'E/13', 'E/06', 'E/16', 'F/03', 'E/09', 'E/10', 'F/04', 'F/05', 'E/05', 'E/04', 'F/09', 'F/13', 'F/11', 'F/10', 'F/12', 'F/14', 'G/03', 'F/06', 'F/15', 'F/18', 'G/11', 'G/10', 'G/07', 'F/16', 'F/17', 'F/07', 'H/09', 'H/07', 'H/13', 'H/12', 'H/10', 'H/11', 'G/09', 'H/08', 'H/14', 'H/18', 'I/03', 'H/16', 'H/15', 'H/17', 'I/05', 'I/04', 'I/08', 'I/07', 'I/09', 'I/12', 'I/18', 'I/13', 'I/16', 'I/14', 'I/15', 'F/08', 'J/13', 'J/09', 'J/10', 'J/11', 'J/12', 'J/16', 'J/17', 'K/03', 'K/05', 'K/04', 'J/08', 'K/06', 'K/08', 'K/07', 'K/09', 'K/11', 'K/10', 'K/13', 'K/15', 'K/17', 'K/18', 'L/03', 'L/04', 'L/05', 'L/06', 'L/07', 'L/10', 'L/11', 'J/18', 'L/12', 'L/14', 'L/15', 'L/16', 'K/12', 'L/17', 'L/18', 'L/13', 'M/03', 'M/05', 'J/05', 'J/15', 'J/14', 'K/16', 'I/06', 'G/08', 'G/06', 'M/10', 'L/09', 'M/13', 'G/15', 'M/14', 'M/16', 'M/15', 'M/17', 'L/08', 'M/04', 'H/06', 'M/18', 'H/05', 'M/06', 'G/16', 'J/07', 'K/14', 'I/11', 'J/06', 'G/13', 'H/04']
Viewing well: E/03


## Whole plate at once, laid out like a physical plate

`viewer.add_plate(PLATE_PATH)` builds a lazy, full-resolution pyramid for every
populated well at its true row/column grid position and adds it to the viewer --
one `Image` layer per channel, a `Well_ROI_table` outline for every well (visible by
default), a `Labels` layer per label found on any well (hidden by default), and an
`FOV_ROI_table` outline for every field of view (hidden by default -- turn it on once
zoomed into a single well). Point-object tables are not included here -- that level of
per-object detail belongs in the per-well view below.

Each `Labels` layer's own measurements are *not* merged/attached here -- reading a
label's plate-wide feature table means fetching its full AnnData-backed table (every
column, every contributing well), a real cost worth paying only for the label(s) you
actually want to inspect. Call `viewer.attach_plate_wide_measurements(PLATE_PATH,
label_name)` afterward for those -- see the next cell -- keyed to match the layer's
plate-wide-unique pixel values so it's usable directly with a tool like
[napari-feature-visualization](https://github.com/fractal-napari-plugins-collection/napari-feature-visualization).
Since raw label IDs are only unique *within* one well, each well's own IDs get a
well-specific offset first (`blimp.ome_ngff.labels.well_label_offset`) so nothing
collides at plate scale -- this promotes the layer's data to `int64`.

This stays fast and full resolution regardless of how sparse the plate is:
`blimp.ome_ngff.plate.build_plate_pyramid` builds the canvas as a `dask.array.zeros`
placeholder (defined analytically -- dask never touches individual chunks just to
construct one) and overlays only populated wells' real data via chunk-aligned
assignment, so cost scales with how many wells actually have data, not with the
plate's declared grid size. A small gap (5% of tile size, computed per pyramid level)
is left between adjacent wells so touching wells stay visually distinct.

In [2]:
plate_viewer = add_blimp_napari_methods(napari.Viewer())
plate_viewer.add_plate(str(PLATE_PATH))

[<Image layer '488' at 0x32caf6310>,
 <Image layer '561' at 0x32cae5410>,
 <Image layer '647' at 0x32ab4d150>,
 <Image layer '405' at 0x333704450>,
 <Labels layer 'Nuclei' at 0x334d2cc50>,
 <Shapes layer 'Well_ROI_table' at 0x332b1fb10>,
 <Shapes layer 'FOV_ROI_table' at 0x333b7ef10>]

## Attach a label's plate-wide measurements (for hover-inspection)

`add_plate` above doesn't merge/attach any label's own measurements automatically --
a real, request-heavy cost (over a remote store) worth paying only for the label(s)
you actually want to inspect. `attach_plate_wide_measurements` does exactly what
`add_plate` used to do eagerly for every label, but on demand, for just the one you
name -- it finds the already-added `Labels` layer and sets its `.features`.

In [ ]:
plate_viewer.attach_plate_wide_measurements(str(PLATE_PATH), "Nuclei")

## Plate-wide feature heatmap

Coloring the plate-wide `Labels` layer above by a feature (e.g. with
[napari-feature-visualization](https://github.com/fractal-napari-plugins-collection/napari-feature-visualization))
can crash: that plugin builds a dense lookup array sized to the *largest* label ID
present, and our plate-wide IDs (`well_label_offset`-scaled to stay unique across
every well) can run into the trillions across a full plate -- fine for one well's own
compact local IDs, not for this.

`viewer.add_feature_heatmap(PLATE_PATH, label_name, feature_name)` sidesteps this
entirely: it builds an ordinary continuous-colormap `Image` layer where each object's
own pixels hold its own measurement directly (a plain float, `NaN` for background or
unmeasured objects) -- no per-ID colormap lookup involved at all. This is a genuinely
new layer, added alongside the `Labels` layer above (which stays exactly as it was) --
the per-well view further down is where Napari Feature Visualizer remains a good fit,
since one well's own local IDs are small.

Not sure which `feature_name` values are available? Merge every well's own features
table the same way `add_feature_heatmap` does, and look at its columns:

```python
from blimp.ome_ngff.plate import _read_plate_wide_features

features_df = _read_plate_wide_features(PLATE_PATH, "Nuclei", kind="mip")
print(features_df.columns.tolist())
```

Building the heatmap costs roughly one pass over every *populated* well's own label
pixels, not the plate's declared grid size -- but for a large, fully-populated plate
this can still take a couple of minutes if you ask for every well at once. Pass
`wells="C/09"` (or a list of well paths) to restrict this to only the well(s) you
actually need.

In [4]:
from blimp.ome_ngff.plate import _read_plate_wide_features

features_df = _read_plate_wide_features(PLATE_PATH, "Nuclei", kind="mip")
print(features_df.columns.tolist())

['label', 'Nuclei_centroid_0', 'Nuclei_centroid_1', 'Nuclei_area', 'Nuclei_area_convex', 'Nuclei_axis_major_length', 'Nuclei_axis_minor_length', 'Nuclei_eccentricity', 'Nuclei_extent', 'Nuclei_feret_diameter_max', 'Nuclei_solidity', 'Nuclei_perimeter', 'Nuclei_perimeter_crofton', 'Nuclei_is_border', 'Nuclei_intensity_mean_None', 'Nuclei_intensity_max_None', 'Nuclei_intensity_min_None', 'Nuclei_intensity_sd_None', 'Nuclei_intensity_median_None', 'Nuclei_intensity_sum_None', 'Nuclei_intensity_mean_488', 'Nuclei_intensity_max_488', 'Nuclei_intensity_min_488', 'Nuclei_intensity_sd_488', 'Nuclei_intensity_median_488', 'Nuclei_intensity_sum_488', 'Nuclei_intensity_mean_561', 'Nuclei_intensity_max_561', 'Nuclei_intensity_min_561', 'Nuclei_intensity_sd_561', 'Nuclei_intensity_median_561', 'Nuclei_intensity_sum_561', 'Nuclei_intensity_mean_XXX', 'Nuclei_intensity_max_XXX', 'Nuclei_intensity_min_XXX', 'Nuclei_intensity_sd_XXX', 'Nuclei_intensity_median_XXX', 'Nuclei_intensity_sum_XXX', 'Nuclei_i

In [5]:
plate_viewer.add_feature_heatmap(str(PLATE_PATH), "Nuclei", "Nuclei_intensity_mean_488")

<Labels layer 'Nuclei: Nuclei_intensity_mean_488' at 0x42692cc50>

## Per-well detail: image, labels, measurements, ROIs

Hands the well's image group to the `napari-ome-zarr` plugin reader directly.
OME-NGFF's multiscale pyramid and channel colors are core spec, not a blimp-specific
convention -- the plugin already reads both correctly and lazily (dask-backed, so
napari picks whichever pyramid level fits the current zoom instead of decoding the
full-resolution array up front). It even auto-discovers any labels as plain `Labels`
layers, since OME-NGFF labels are core spec too -- see the next cell for attaching
their measurements.

In [ ]:
for kind in ("mip", "stack"):
    image_group_path = PLATE_PATH / WELL / kind
    if image_group_path.exists():
        break
else:
    raise FileNotFoundError(f"No mip or stack image found for well {WELL}")

viewer = add_blimp_napari_methods(napari.Viewer())
layers = viewer.open(str(image_group_path), plugin="napari-ome-zarr")
print(f"Opened '{kind}':", [layer.name for layer in layers])

## Add measurements, point objects, and FOV boundaries

The cell above's `Labels` layers (if any) are bare pixel data -- napari-ome-zarr has no
idea blimp attached a linked `FeatureTable` to each one, since that's a blimp/Fractal/ngio
convention, not core OME-NGFF spec. Rather than reading the label a second time via
`blimp.napari_utils.add_labels_with_measurements`, attach each one's measurements
directly onto the layer already loaded above.

Point-object tables (spots/blobs with no stable pixel identity -- see
`blimp.ome_ngff.labels._write_well_points`) and FOV boundaries genuinely have no core-spec
equivalent at all, so those still go through blimp's own helpers. A well with none of
these just adds nothing here.

In [ ]:
container = open_ome_zarr_container(str(image_group_path))

for layer in layers:
    if isinstance(layer, napari.layers.Labels):
        label_name = layer.name.rsplit("/", 1)[-1]  # napari-ome-zarr nests it under "labels/labels/<name>"
        table_name = f"{label_name}_features"
        if table_name in container.list_tables():
            layer.features = container.get_feature_table(table_name).dataframe.reset_index()
            print(f"Attached {len(layer.features)} rows of measurements to '{layer.name}'")

for table_name in container.list_tables():
    if container.get_table(table_name).table_type() == "generic_roi_table":
        viewer.add_points_with_measurements(image_group_path, table_name)

if "FOV_ROI_table" in container.list_tables():
    viewer.add_rois(image_group_path)